# Olefin Hydroformylation Agent — Results Analysis

This notebook visualises the performance of the agentic LLM optimizer across experimental iterations. Run all cells top-to-bottom after at least one complete agent run (`python src/agent.py`).

**Final Project for CSC 7644: Applied LLM Development**  
Author: Chidera C. Nnadiekwe

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make src/ importable
sys.path.insert(0, str(Path('..') / 'src'))
sys.path.insert(0, str(Path('..') / 'scripts'))

LOG_PATH = Path('..') / 'data' / 'experiment_log.json'

with open(LOG_PATH) as f:
    log = json.load(f)

print(f"Loaded {len(log['runs'])} experimental runs.")

## 1. Tabular Summary of All Runs

In [ ]:
rows = []
for run in log['runs']:
    row = {'iteration': run['iteration']}
    row.update(run['conditions'])
    row.update(run['outcomes'])
    rows.append(row)

df = pd.DataFrame(rows)
df.set_index('iteration', inplace=True)
df.style.highlight_max(subset=['lb_ratio', 'aldehyde_yield_pct'], color='#d4edda')

## 2. L:B Selectivity Over Iterations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df.index, df['lb_ratio'], 'o-', color='#2563eb', linewidth=2, markersize=7, label='L:B ratio')
ax.axhline(y=df['lb_ratio'].max(), color='#2563eb', linestyle='--', alpha=0.4, label=f'Best: {df["lb_ratio"].max():.2f}')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('L:B Selectivity Ratio', fontsize=12)
ax.set_title('Linear-to-Branch Selectivity vs Iteration', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/lb_convergence.png', dpi=150)
plt.show()

## 3. Aldehyde Yield Over Iterations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(df.index, df['aldehyde_yield_pct'], alpha=0.15, color='#16a34a')
ax.plot(df.index, df['aldehyde_yield_pct'], 'o-', color='#16a34a', linewidth=2, markersize=7)
ax.set_ylim(0, 100)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Aldehyde Yield (%)', fontsize=12)
ax.set_title('Aldehyde Yield vs Iteration', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/yield_convergence.png', dpi=150)
plt.show()

## 4. Parameter Exploration Heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

params = ['temperature_C', 'pressure_bar', 'ligand_equiv']
xlabels = ['Temperature (°C)', 'Pressure (bar)', 'Ligand Equiv']
colors = ['#7c3aed', '#0891b2', '#ea580c']

for ax, param, xlabel, color in zip(axes, params, xlabels, colors):
    sc = ax.scatter(df[param], df['lb_ratio'], c=df.index,
                    cmap='viridis', s=80, alpha=0.85, edgecolors='k', linewidths=0.5)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('L:B Ratio', fontsize=11)
    ax.set_title(f'{xlabel} vs L:B', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.colorbar(sc, ax=ax, label='Iteration')

plt.suptitle('Parameter–Selectivity Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/parameter_lb_scatter.png', dpi=150)
plt.show()

## 5. Agent Reasoning Traces

In [ ]:
for run in log['runs']:
    print(f"=== Iteration {run['iteration']} ===")
    print(f"L:B: {run['outcomes']['lb_ratio']} | Yield: {run['outcomes']['aldehyde_yield_pct']}%")
    print(f"Reasoning trace (first 400 chars):")
    print(run['reasoning_trace'][:400])
    print()

## 6. Summary Metrics

In [ ]:
print("\n=== SUMMARY ===")
print(f"Total iterations: {len(df)}")
print(f"Best L:B ratio:   {df['lb_ratio'].max():.2f} (iteration {df['lb_ratio'].idxmax()})")
print(f"Best yield:       {df['aldehyde_yield_pct'].max():.1f}% (iteration {df['aldehyde_yield_pct'].idxmax()})")
print(f"Mean L:B ± std:   {df['lb_ratio'].mean():.2f} ± {df['lb_ratio'].std():.2f}")

for threshold in [3.0, 4.0, 5.0]:
    idx = df[df['lb_ratio'] >= threshold].index
    if len(idx) > 0:
        print(f"Iterations to L:B ≥ {threshold}: {idx[0]}")
    else:
        print(f"L:B ≥ {threshold} not achieved in {len(df)} iterations")